In [ ]:
# My Text-to-Image Project

# This notebook is a simple project for generating images from text prompts with a mix of Stable Diffusion, LoRA, embeddings, and GAN-based ideas.

# The flow is:
# 1. Install the needed packages
# 2. Load the project modules
# 3. Try text embedding and prompt analysis
# 4. Generate a few shapes with a CGAN
# 5. Try the attention-based GAN
# 6. Run the full pipeline demo
# 7. Launch the Gradio interface

In [ ]:
import importlib.util
import subprocess
import sys


def ensure_package(module_name, pip_name=None):
    pip_name = pip_name or module_name
    if importlib.util.find_spec(module_name) is None:
        print(f"Installing {pip_name}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pip_name])


ensure_package("torch")
ensure_package("torchvision")
ensure_package("torchaudio")
ensure_package("diffusers")
ensure_package("accelerate")
ensure_package("peft")
ensure_package("transformers")
ensure_package("scipy")
ensure_package("gradio")
ensure_package("matplotlib")
ensure_package("PIL")
ensure_package("numpy")

import os
import warnings
warnings.filterwarnings("ignore")

print("Setup complete")

In [ ]:
from app import StableDiffusionGenerator, StableDiffusionUI
from cgan import CGANModel
from fine_tuner import LoRAFineTuner
from prompt_encoder import PromptEncoder
from attention_gan import AttentionGANModel
from pipeline import ImageGenerationPipeline, BatchPipelineProcessor

print("Project modules loaded successfully")

In [ ]:
sample_prompt = "a red circle on a white background"

try:
    enc = PromptEncoder(load_model=False)
    result = enc.encode(sample_prompt)
    print("Prompt:", sample_prompt)
    print("Embedding shape:", result["last_hidden_state"].shape)
    print("Pooled shape:", result["pooled_output"].shape)
    print("Token count:", result["n_tokens"])
    print("Fallback used:", result.get("fallback_used", False))
except Exception as e:
    print("CLIP download did not finish, so I am using the lightweight fallback demo.")
    from pipeline import TextPreprocessor
    pre = TextPreprocessor(load_text_model=False)
    analysis = pre.analyze_text(sample_prompt)
    print("Fallback embedding dim:", analysis["embedding"]["embedding_dim"])
    print("Fallback used:", analysis["embedding"]["fallback_used"])


# CGAN Shape Demo

This section loads the pre-trained conditional GAN and generates a few shapes from the saved weights.

In [ ]:
import os

cgan_model = CGANModel()
cgan_model.load_weights()

os.makedirs("outputs", exist_ok=True)

if cgan_model.is_trained:
    sample = cgan_model.generate(0)  # square
    sample.save("outputs/cgan_square_demo.png")
    print("Saved CGAN sample to outputs/cgan_square_demo.png")
    sample.show()
else:
    print("CGAN weights could not be loaded")

In [ ]:
# Attention GAN Demo

import os

os.makedirs("outputs", exist_ok=True)

attention_model = AttentionGANModel()
attention_model.is_trained = True

sample = attention_model.generate(1)  # circle
sample.save("outputs/attention_circle_demo.png")
print("Saved attention GAN sample to outputs/attention_circle_demo.png")
sample.show()

In [ ]:
# Full Pipeline Demo

pipeline = ImageGenerationPipeline(
    enable_sd=False,
    enable_cgan=True,
    enable_attention_gan=False,
    load_heavy_models=False,
)

result = pipeline.analyze_and_generate(
    prompt="a beautiful sunset over the ocean",
    model_choice="cgan"
)

print("Image path:", result.get("image_path"))
print("Metadata path:", result.get("metadata_path"))
print("Text analysis ready:", result["text_analysis"]["character_count"])


In [ ]:
# Launch the UI (disabled by default for fast notebook execution)

print("Skipping the interactive Gradio UI launch in this fast demo mode.")
print("Uncomment the lines below if you want to start the UI manually.")
# ui = StableDiffusionUI()
# interface = ui.create_interface()
# interface.launch(share=True, server_name="0.0.0.0", server_port=7860, debug=True, show_error=True)